# Data Structures — Dictionaries — Problems and Solutions  
**Technical Notes**  
*Rodrigo Kang*

This notebook contains the solutions to the problems included in the associated technical notes on Python dictionaries.

The problems are designed to reinforce not only dictionary syntax, but also the reasoning required to understand **mapping semantics, key lookup, hashability, views, mutability, aliasing, copying, merging, nested dictionaries, and dictionary comprehensions**.

The emphasis is therefore on predicting behaviour, explaining why it occurs, and choosing operations whose semantics match the intended data model.

## Problem 1 — Basic Dictionary Access

### Problem

Given

```python
model = {
    "name": "linear",
    "score": 0.84,
    "features": 12,
}
```

determine the result of

```python
model["name"]
model["score"]
len(model)
```

What does `len(model)` count?

### Solution

The expressions return:

```text
model["name"]  -> "linear"
model["score"] -> 0.84
len(model)     -> 3
```

`len(model)` counts the number of **top-level key-value associations** in the dictionary. It does not count objects nested inside the values.

#### Verification

In [1]:
model = {
    "name": "linear",
    "score": 0.84,
    "features": 12,
}

model["name"], model["score"], len(model)

('linear', 0.84, 3)

### Key Takeaway

Dictionary length counts entries in the mapping, while indexed access retrieves the value associated with a key.

## Problem 2 — Missing Keys

### Problem

Given

```python
metrics = {
    "accuracy": 0.91,
}
```

compare

```python
metrics["recall"]
metrics.get("recall")
metrics.get("recall", 0.0)
```

Explain the different missing-key semantics.

### Solution

Direct access with

```python
metrics["recall"]
```

raises `KeyError` because the key is required to exist.

By contrast,

```python
metrics.get("recall")
```

returns `None`, and

```python
metrics.get("recall", 0.0)
```

returns the explicitly provided fallback `0.0`.

The difference is semantic: bracket lookup treats absence as an error, while `get()` allows absence to be represented by a fallback value.

#### Verification

In [2]:
metrics = {"accuracy": 0.91}

try:
    print(metrics["recall"])
except KeyError as exc:
    print(type(exc).__name__, exc)

print(metrics.get("recall"))
print(metrics.get("recall", 0.0))

KeyError 'recall'
None
0.0


### Key Takeaway

Use bracket lookup when absence is invalid; use `get()` when a missing key is an expected possibility.

## Problem 3 — Membership

### Problem

Given

```python
record = {
    "name": "model_a",
    "score": 0.91,
}
```

predict:

```python
"name" in record
"model_a" in record
0.91 in record
```

Explain what membership tests on a dictionary actually inspect.

### Solution

Dictionary membership tests inspect **keys**, not values.

Therefore:

```text
"name" in record      -> True
"model_a" in record   -> False
0.91 in record        -> False
```

The latter two objects are values, not keys.

#### Verification

In [3]:
record = {
    "name": "model_a",
    "score": 0.91,
}

"name" in record, "model_a" in record, 0.91 in record

(True, False, False)

### Key Takeaway

`x in dictionary` means “is `x` a key in this mapping?”.

## Problem 4 — Stored `None` versus Missing Key

### Problem

Consider:

```python
record = {
    "comment": None,
}
```

Compare:

```python
record.get("comment")
record.get("owner")
```

Why are the return values insufficient to determine whether the key exists?

Write an appropriate test for key presence.

### Solution

Both expressions return `None`:

```python
record.get("comment")  # existing key with value None
record.get("owner")    # missing key, default None
```

Therefore, the returned value alone cannot distinguish the two states.

If presence itself matters, test the key explicitly:

```python
"comment" in record
"owner" in record
```

#### Verification

In [4]:
record = {"comment": None}

print(record.get("comment"))
print(record.get("owner"))
print("comment" in record)
print("owner" in record)

None
None
True
False


### Key Takeaway

A fallback value can be indistinguishable from a legitimate stored value; membership preserves the difference between presence and absence.

## Problem 5 — Duplicate Keys

### Problem

Predict:

```python
data = {
    "x": 1,
    "y": 2,
    "x": 3,
}
```

What is the final value associated with `"x"`?

Why are there not two separate `"x"` entries?

### Solution

The resulting dictionary is effectively

```python
{"x": 3, "y": 2}
```

and

```python
data["x"] == 3
```

A dictionary is a mapping in which each key is unique. When the same key is specified again, the later value replaces the earlier association rather than creating another entry with the same key.

#### Verification

In [5]:
data = {
    "x": 1,
    "y": 2,
    "x": 3,
}

data, data["x"]

({'x': 3, 'y': 2}, 3)

### Key Takeaway

A dictionary key identifies at most one current value; later assignments for the same key overwrite earlier ones.

## Problem 6 — Valid and Invalid Keys

### Problem

Determine which of the following objects can be dictionary keys:

```python
42
3.14
"model"
(1, 2)
[1, 2]
{"x": 1}
```

Explain the criterion using **hashability** rather than merely listing allowed types.

### Solution

The first four objects are hashable and can be keys:

```text
42
3.14
"model"
(1, 2)
```

The list and dictionary are mutable and unhashable:

```text
[1, 2]
{"x": 1}
```

Attempting to hash them raises `TypeError`.

The actual dictionary requirement is **hashability**: a key must have a stable hash and equality behaviour suitable for hash-based lookup.

#### Verification

In [6]:
objects = [42, 3.14, "model", (1, 2), [1, 2], {"x": 1}]

for obj in objects:
    try:
        print(repr(obj), "-> hash:", hash(obj))
    except TypeError as exc:
        print(repr(obj), "->", type(exc).__name__)

42 -> hash: 42
3.14 -> hash: 322818021289917443
'model' -> hash: 1993442803039093723
(1, 2) -> hash: -3550055125485641917
[1, 2] -> TypeError
{'x': 1} -> TypeError


### Key Takeaway

Dictionary keys must be hashable; immutability is relevant because stable hashing is required for lookup.

## Problem 7 — Updating and Inserting

### Problem

Starting from

```python
config = {
    "epochs": 100,
    "learning_rate": 0.01,
}
```

use indexed assignment to

1. change `epochs` to `200`;
2. insert `"batch_size": 64`.

Explain why the same syntax performs both operations.

### Solution

The required assignments are

```python
config["epochs"] = 200
config["batch_size"] = 64
```

For an existing key, assignment updates the associated value. For a missing key, assignment inserts a new key-value pair.

#### Verification

In [7]:
config = {
    "epochs": 100,
    "learning_rate": 0.01,
}

config["epochs"] = 200
config["batch_size"] = 64

config

{'epochs': 200, 'learning_rate': 0.01, 'batch_size': 64}

### Key Takeaway

Dictionary assignment is defined by the key: it updates an existing association or creates a new one when the key is absent.

## Problem 8 — Dictionary Equality

### Problem

Predict:

```python
a = {"x": 1, "y": 2}
b = {"y": 2, "x": 1}

a == b
```

Then compare this with:

```python
[1, 2] == [2, 1]
```

Explain the difference between mapping equality and sequence equality.

### Solution

The dictionaries compare equal:

```python
a == b  # True
```

because they contain the same key-value mapping, even though the insertion orders differ.

The lists compare unequal:

```python
[1, 2] == [2, 1]  # False
```

because sequence equality is order-sensitive.

Dictionary equality concerns associations; list equality concerns ordered positions.

#### Verification

In [8]:
a = {"x": 1, "y": 2}
b = {"y": 2, "x": 1}

print(a == b)
print([1, 2] == [2, 1])

True
False


### Key Takeaway

Mappings compare by key-value associations; sequences compare by corresponding positions.

## Problem 9 — Iterating Directly

### Problem

Given

```python
metrics = {
    "accuracy": 0.91,
    "precision": 0.87,
    "recall": 0.84,
}
```

what values does

```python
for x in metrics:
    print(x)
```

produce?

Rewrite the loop explicitly using the equivalent dictionary view.

### Solution

Direct iteration traverses dictionary keys, in insertion order:

```text
accuracy
precision
recall
```

The explicit equivalent is

```python
for x in metrics.keys():
    print(x)
```

#### Verification

In [9]:
metrics = {
    "accuracy": 0.91,
    "precision": 0.87,
    "recall": 0.84,
}

for x in metrics:
    print(x)

print("---")

for x in metrics.keys():
    print(x)

accuracy
precision
recall
---
accuracy
precision
recall


### Key Takeaway

Direct dictionary iteration is key iteration.

## Problem 10 — `keys()`, `values()`, and `items()`

### Problem

For the dictionary in Problem 9, explain what is produced conceptually by

```python
metrics.keys()
metrics.values()
metrics.items()
```

Which method is most appropriate when both the metric name and value are required?

### Solution

Conceptually:

```text
keys()   -> "accuracy", "precision", "recall"
values() -> 0.91, 0.87, 0.84
items()  -> ("accuracy", 0.91), ("precision", 0.87), ("recall", 0.84)
```

When both components are needed, `items()` is the natural choice:

```python
for metric, value in metrics.items():
    ...
```

It provides each association directly as a pair.

#### Verification

In [10]:
metrics = {
    "accuracy": 0.91,
    "precision": 0.87,
    "recall": 0.84,
}

print(list(metrics.keys()))
print(list(metrics.values()))
print(list(metrics.items()))

['accuracy', 'precision', 'recall']
[0.91, 0.87, 0.84]
[('accuracy', 0.91), ('precision', 0.87), ('recall', 0.84)]


### Key Takeaway

Use the dictionary view that matches the information required by the computation.

## Problem 11 — Dynamic Views

### Problem

Predict:

```python
data = {
    "a": 1,
    "b": 2,
}

keys = data.keys()

data["c"] = 3

print(list(keys))
```

Why does `keys` contain `"c"` even though it was created before `"c"` was inserted?

### Solution

The output is

```python
['a', 'b', 'c']
```

`data.keys()` returns a dynamic **view** of the dictionary rather than a detached list. The view remains connected to the underlying mapping and reflects later structural changes.

#### Verification

In [11]:
data = {"a": 1, "b": 2}

keys = data.keys()
data["c"] = 3

print(list(keys))

['a', 'b', 'c']


### Key Takeaway

Dictionary views are live representations of the underlying mapping, not snapshots.

## Problem 12 — Detached List versus View

### Problem

Compare:

```python
data = {"a": 1, "b": 2}

view = data.keys()
snapshot = list(data.keys())

data["c"] = 3
```

Predict

```python
list(view)
snapshot
```

Explain the semantic difference.

### Solution

After inserting `"c"`:

```python
list(view) == ["a", "b", "c"]
snapshot   == ["a", "b"]
```

`view` is dynamically linked to the dictionary. `snapshot` is an independently created list containing only the keys that existed when conversion occurred.

#### Verification

In [12]:
data = {"a": 1, "b": 2}

view = data.keys()
snapshot = list(data.keys())

data["c"] = 3

print(list(view))
print(snapshot)

['a', 'b', 'c']
['a', 'b']


### Key Takeaway

Converting a view to a list captures a separate snapshot of the current contents.

## Problem 13 — Iterating over Items

### Problem

Given

```python
scores = {
    "linear": 0.82,
    "tree": 0.78,
    "forest": 0.91,
}
```

use `items()` to print

```text
linear 0.82
tree 0.78
forest 0.91
```

without performing a second dictionary lookup inside the loop.

### Solution

The key-value pairs can be unpacked directly:

```python
for model, score in scores.items():
    print(model, score)
```

There is no need to iterate over keys and then access `scores[model]` separately.

#### Verification

In [13]:
scores = {
    "linear": 0.82,
    "tree": 0.78,
    "forest": 0.91,
}

for model, score in scores.items():
    print(model, score)

linear 0.82
tree 0.78
forest 0.91


### Key Takeaway

`items()` directly exposes the two components of each mapping entry.

## Problem 14 — `update()`

### Problem

Starting from

```python
config = {
    "epochs": 100,
    "learning_rate": 0.01,
}
```

apply

```python
{
    "epochs": 200,
    "batch_size": 64,
}
```

using `update()`.

Which entry is replaced and which is inserted?

### Solution

Use:

```python
config.update({
    "epochs": 200,
    "batch_size": 64,
})
```

`"epochs"` already exists, so its value is replaced. `"batch_size"` is absent, so a new entry is inserted.

#### Verification

In [14]:
config = {
    "epochs": 100,
    "learning_rate": 0.01,
}

config.update({
    "epochs": 200,
    "batch_size": 64,
})

config

{'epochs': 200, 'learning_rate': 0.01, 'batch_size': 64}

### Key Takeaway

`update()` applies multiple key assignments using the same insert-or-replace rule as ordinary dictionary assignment.

## Problem 15 — Merging Dictionaries

### Problem

Predict:

```python
defaults = {
    "epochs": 100,
    "learning_rate": 0.01,
}

custom = {
    "epochs": 300,
    "batch_size": 64,
}

config = defaults | custom
```

What is `config`?

Are `defaults` and `custom` modified?

Explain the conflict-resolution rule.

### Solution

`config` becomes

```python
{
    "epochs": 300,
    "learning_rate": 0.01,
    "batch_size": 64,
}
```

Neither input dictionary is modified.

When a key appears in both operands, the right-hand dictionary wins. Therefore `"epochs": 300` overrides `"epochs": 100`.

#### Verification

In [15]:
defaults = {
    "epochs": 100,
    "learning_rate": 0.01,
}

custom = {
    "epochs": 300,
    "batch_size": 64,
}

config = defaults | custom

print(config)
print(defaults)
print(custom)

{'epochs': 300, 'learning_rate': 0.01, 'batch_size': 64}
{'epochs': 100, 'learning_rate': 0.01}
{'epochs': 300, 'batch_size': 64}


### Key Takeaway

The merge operator creates a new mapping, with right-hand values resolving key conflicts.

## Problem 16 — `|` versus `|=`

### Problem

Starting from

```python
a = {"x": 1}
b = a
```

compare:

```python
a = a | {"y": 2}
```

with

```python
a |= {"y": 2}
```

using independent starting states.

How does aliasing affect the value observed through `b`?

### Solution

With

```python
a = a | {"y": 2}
```

a new dictionary is created and `a` is rebound to it. `b` still refers to the original:

```text
a -> {"x": 1, "y": 2}
b -> {"x": 1}
```

With

```python
a |= {"y": 2}
```

the original dictionary is updated in place, so both aliases observe

```python
{"x": 1, "y": 2}
```

#### Verification

In [16]:
a = {"x": 1}
b = a
a = a | {"y": 2}
print("merge:", a, b, a is b)

a = {"x": 1}
b = a
a |= {"y": 2}
print("in-place merge:", a, b, a is b)

merge: {'x': 1, 'y': 2} {'x': 1} False
in-place merge: {'x': 1, 'y': 2} {'x': 1, 'y': 2} True


### Key Takeaway

New-object merging and in-place merging differ when other names alias the original dictionary.

## Problem 17 — Removing with `pop()`

### Problem

Given

```python
config = {
    "epochs": 100,
    "batch_size": 64,
}
```

predict:

```python
removed = config.pop("batch_size")
```

What are `removed` and `config` afterwards?

Then explain the role of a default argument to `pop()`.

### Solution

`pop("batch_size")` removes the entry and returns its value:

```text
removed == 64
config == {"epochs": 100}
```

A default argument changes missing-key behaviour:

```python
config.pop("missing", None)
```

returns `None` instead of raising `KeyError`.

#### Verification

In [17]:
config = {
    "epochs": 100,
    "batch_size": 64,
}

removed = config.pop("batch_size")

print(removed)
print(config)
print(config.pop("missing", None))

64
{'epochs': 100}
None


### Key Takeaway

`pop()` is both a lookup and a removal operation; its optional default controls missing-key behaviour.

## Problem 18 — `popitem()`

### Problem

Given

```python
data = {
    "a": 1,
    "b": 2,
    "c": 3,
}
```

predict the pair removed by

```python
data.popitem()
```

Explain how insertion order determines the result.

### Solution

Modern Python dictionaries preserve insertion order, and `popitem()` removes the most recently inserted pair.

Therefore it returns:

```python
("c", 3)
```

and leaves

```python
{"a": 1, "b": 2}
```

#### Verification

In [18]:
data = {
    "a": 1,
    "b": 2,
    "c": 3,
}

removed = data.popitem()
removed, data

(('c', 3), {'a': 1, 'b': 2})

### Key Takeaway

`popitem()` follows last-in, first-out removal based on dictionary insertion order.

## Problem 19 — `clear()` and Aliasing

### Problem

Predict:

```python
a = {"x": 1, "y": 2}
b = a

a.clear()

print(a)
print(b)
```

Then compare with:

```python
a = {"x": 1, "y": 2}
b = a

a = {}

print(a)
print(b)
```

Explain mutation versus rebinding.

### Solution

In the first case, `clear()` mutates the shared dictionary, so both names display `{}`.

In the second case, `a = {}` rebinds only `a` to a newly created empty dictionary. `b` remains attached to the original mapping:

```text
a == {}
b == {"x": 1, "y": 2}
```

This is the distinction between mutation and rebinding.

#### Verification

In [19]:
a = {"x": 1, "y": 2}
b = a
a.clear()
print(a)
print(b)

a = {"x": 1, "y": 2}
b = a
a = {}
print(a)
print(b)

{}
{}
{}
{'x': 1, 'y': 2}


### Key Takeaway

Mutating a shared object is visible through every alias; rebinding changes only one name.

## Problem 20 — `setdefault()`

### Problem

Predict:

```python
config = {
    "epochs": 100,
}

a = config.setdefault("epochs", 200)
b = config.setdefault("batch_size", 64)
```

Determine `a`, `b`, and the final dictionary.

### Solution

For `"epochs"`, the key already exists, so its current value is returned and no update occurs:

```python
a == 100
```

For `"batch_size"`, the key is absent, so it is inserted with `64` and that value is returned:

```python
b == 64
```

Final dictionary:

```python
{"epochs": 100, "batch_size": 64}
```

#### Verification

In [20]:
config = {"epochs": 100}

a = config.setdefault("epochs", 200)
b = config.setdefault("batch_size", 64)

a, b, config

(100, 64, {'epochs': 100, 'batch_size': 64})

### Key Takeaway

`setdefault()` returns an existing value or inserts and returns a default only when the key is absent.

## Problem 21 — Aliasing

### Problem

Without executing the code, determine the final values:

```python
a = {"score": 0.81}
b = a

b["score"] = 0.91
b["name"] = "model_a"
```

What is `a is b`?

### Solution

`b = a` creates an alias. Both names refer to the same mutable dictionary.

The final mapping is

```python
{
    "score": 0.91,
    "name": "model_a",
}
```

through both `a` and `b`, and

```python
a is b
```

is `True`.

#### Verification

In [21]:
a = {"score": 0.81}
b = a

b["score"] = 0.91
b["name"] = "model_a"

a, b, a is b

({'score': 0.91, 'name': 'model_a'}, {'score': 0.91, 'name': 'model_a'}, True)

### Key Takeaway

Assignment does not copy a dictionary; it creates another reference to the same object.

## Problem 22 — Shallow Copy

### Problem

Predict:

```python
a = {
    "score": 0.81,
}

b = a.copy()

b["score"] = 0.91
```

What are the two dictionaries?

What are

```python
a == b
a is b
```

after the modification?

### Solution

`copy()` creates a distinct outer dictionary. Replacing a value in `b` therefore does not affect `a`.

```text
a == {"score": 0.81}
b == {"score": 0.91}
```

After the change:

```text
a == b -> False
a is b -> False
```

#### Verification

In [22]:
a = {"score": 0.81}
b = a.copy()

b["score"] = 0.91

a, b, a == b, a is b

({'score': 0.81}, {'score': 0.91}, False, False)

### Key Takeaway

A shallow copy separates the outer mapping, so replacing top-level values is independent.

## Problem 23 — Shallow Copy with Mutable Values

### Problem

Consider:

```python
a = {
    "features": ["age", "income"],
}

b = a.copy()

b["features"].append("balance")
```

Predict both dictionaries.

Then determine:

```python
a is b
a["features"] is b["features"]
```

### Solution

The outer mappings are distinct, but their `"features"` values refer to the same list.

Therefore both display:

```python
{"features": ["age", "income", "balance"]}
```

Identity:

```text
a is b                         -> False
a["features"] is b["features"] -> True
```

#### Verification

In [23]:
a = {
    "features": ["age", "income"],
}

b = a.copy()
b["features"].append("balance")

a, b, a is b, a["features"] is b["features"]

({'features': ['age', 'income', 'balance']},
 {'features': ['age', 'income', 'balance']},
 False,
 True)

### Key Takeaway

Shallow copying duplicates the mapping but shares references to nested mutable values.

## Problem 24 — Replacement versus Nested Mutation

### Problem

Starting independently from

```python
a = {
    "features": ["age", "income"],
}

b = a.copy()
```

compare:

```python
b["features"].append("balance")
```

with

```python
b["features"] = ["balance"]
```

Explain why only one of these operations necessarily modifies what is observed through `a`.

### Solution

The first operation mutates the shared list object, so `a["features"]` changes too.

The second operation replaces the value associated with `"features"` only inside `b`. It changes the reference stored in `b` without mutating the original list still referenced by `a`.

Thus mutation follows shared identity; replacement is local to the outer mapping being modified.

#### Verification

In [24]:
a = {"features": ["age", "income"]}
b = a.copy()
b["features"].append("balance")
print("nested mutation:", a, b)

a = {"features": ["age", "income"]}
b = a.copy()
b["features"] = ["balance"]
print("replacement:", a, b)

nested mutation: {'features': ['age', 'income', 'balance']} {'features': ['age', 'income', 'balance']}
replacement: {'features': ['age', 'income']} {'features': ['balance']}


### Key Takeaway

Mutating a shared value and replacing a mapping entry are different operations with different aliasing effects.

## Problem 25 — Deep Copy

### Problem

Given

```python
config = {
    "model": {
        "type": "forest",
        "trees": 100,
    },
    "features": ["age", "income"],
}
```

create an independent recursive copy using `copy.deepcopy()`.

Modify both nested structures in the copy and verify that the original remains unchanged.

### Solution

A deep copy recursively duplicates nested mutable objects:

```python
import copy

copied = copy.deepcopy(config)
copied["model"]["trees"] = 300
copied["features"].append("balance")
```

The original remains unchanged because the nested dictionary and list in the copy have independent identities.

#### Verification

In [25]:
import copy

config = {
    "model": {
        "type": "forest",
        "trees": 100,
    },
    "features": ["age", "income"],
}

copied = copy.deepcopy(config)
copied["model"]["trees"] = 300
copied["features"].append("balance")

print(config)
print(copied)
print(config["model"] is copied["model"])
print(config["features"] is copied["features"])

{'model': {'type': 'forest', 'trees': 100}, 'features': ['age', 'income']}
{'model': {'type': 'forest', 'trees': 300}, 'features': ['age', 'income', 'balance']}
False
False


### Key Takeaway

Use deep copying when recursive independence of nested mutable values is genuinely required.

## Problem 26 — Nested Access

### Problem

Given

```python
experiment = {
    "model": {
        "name": "forest",
        "trees": 200,
    },
    "metrics": {
        "accuracy": 0.91,
        "recall": 0.84,
    },
}
```

retrieve

1. `"forest"`;
2. `200`;
3. `0.84`.

Write each nested-key expression explicitly.

### Solution

The required expressions are:

```python
experiment["model"]["name"]      # "forest"
experiment["model"]["trees"]     # 200
experiment["metrics"]["recall"]  # 0.84
```

Each bracket operation retrieves one level of the nested mapping.

#### Verification

In [26]:
experiment = {
    "model": {
        "name": "forest",
        "trees": 200,
    },
    "metrics": {
        "accuracy": 0.91,
        "recall": 0.84,
    },
}

(
    experiment["model"]["name"],
    experiment["model"]["trees"],
    experiment["metrics"]["recall"],
)

('forest', 200, 0.84)

### Key Takeaway

Nested dictionary access follows the hierarchy one key lookup at a time.

## Problem 27 — Updating Nested Dictionaries

### Problem

Using the dictionary from Problem 26,

1. change `trees` to `300`;
2. insert `"precision": 0.87` into the metrics dictionary.

Do not replace the complete nested dictionaries.

### Solution

Modify the nested mappings directly:

```python
experiment["model"]["trees"] = 300
experiment["metrics"]["precision"] = 0.87
```

The first assignment updates an existing nested association; the second inserts a new nested association.

#### Verification

In [27]:
experiment = {
    "model": {
        "name": "forest",
        "trees": 200,
    },
    "metrics": {
        "accuracy": 0.91,
        "recall": 0.84,
    },
}

experiment["model"]["trees"] = 300
experiment["metrics"]["precision"] = 0.87

experiment

{'model': {'name': 'forest', 'trees': 300},
 'metrics': {'accuracy': 0.91, 'recall': 0.84, 'precision': 0.87}}

### Key Takeaway

Nested dictionaries retain ordinary dictionary mutation semantics at every level.

## Problem 28 — Shallow Merge of Nested Dictionaries

### Problem

Predict:

```python
defaults = {
    "model": {
        "trees": 100,
        "max_depth": 10,
    },
    "verbose": False,
}

custom = {
    "model": {
        "trees": 300,
    },
}

merged = defaults | custom
```

Does `merged["model"]` contain `"max_depth"`?

Explain why ordinary dictionary merge is not recursive.

### Solution

No. The result is

```python
{
    "model": {
        "trees": 300,
    },
    "verbose": False,
}
```

At the top level, both dictionaries contain the key `"model"`. The right-hand value associated with that key replaces the complete left-hand value.

The merge operator resolves top-level key collisions; it does not recursively merge nested mappings.

#### Verification

In [28]:
defaults = {
    "model": {
        "trees": 100,
        "max_depth": 10,
    },
    "verbose": False,
}

custom = {
    "model": {
        "trees": 300,
    },
}

merged = defaults | custom
merged

{'model': {'trees': 300}, 'verbose': False}

### Key Takeaway

Ordinary dictionary merging is shallow with respect to nested mapping values.

## Problem 29 — Building from `zip()`

### Problem

Given

```python
names = ["linear", "tree", "forest"]
scores = [0.82, 0.78, 0.91]
```

construct

```python
{
    "linear": 0.82,
    "tree": 0.78,
    "forest": 0.91,
}
```

using `dict()` and `zip()`.

### Solution

`zip()` forms corresponding key-value pairs and `dict()` consumes them:

```python
result = dict(zip(names, scores))
```

The resulting mapping associates each model name with the score at the corresponding position.

#### Verification

In [29]:
names = ["linear", "tree", "forest"]
scores = [0.82, 0.78, 0.91]

dict(zip(names, scores))

{'linear': 0.82, 'tree': 0.78, 'forest': 0.91}

### Key Takeaway

`dict(zip(keys, values))` is a direct construction pattern for aligned key and value sequences.

## Problem 30 — Mismatched `zip()` Inputs

### Problem

Predict:

```python
names = ["linear", "tree", "forest"]
scores = [0.82, 0.78]

dict(zip(names, scores))
```

Which element is lost?

Why can this be dangerous?

Then explain how `strict=True` changes the behaviour.

### Solution

Default `zip()` stops when the shortest iterable is exhausted, so the result is

```python
{
    "linear": 0.82,
    "tree": 0.78,
}
```

The key `"forest"` is silently discarded.

This can hide alignment errors when equal lengths are expected.

Using

```python
zip(names, scores, strict=True)
```

raises `ValueError` when the lengths differ.

#### Verification

In [30]:
names = ["linear", "tree", "forest"]
scores = [0.82, 0.78]

print(dict(zip(names, scores)))

try:
    print(dict(zip(names, scores, strict=True)))
except ValueError as exc:
    print(type(exc).__name__, exc)

{'linear': 0.82, 'tree': 0.78}
ValueError zip() argument 2 is shorter than argument 1


### Key Takeaway

Use strict parallel iteration when unequal lengths represent invalid data rather than acceptable truncation.

## Problem 31 — Basic Dictionary Comprehension

### Problem

Construct a dictionary mapping the integers from `1` through `5` to their squares.

The expected result is

```python
{
    1: 1,
    2: 4,
    3: 9,
    4: 16,
    5: 25,
}
```

Use a dictionary comprehension.

### Solution

The comprehension is

```python
squares = {
    value: value ** 2
    for value in range(1, 6)
}
```

Each input integer becomes the key, and its square becomes the corresponding value.

#### Verification

In [31]:
squares = {
    value: value ** 2
    for value in range(1, 6)
}

squares

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25}

### Key Takeaway

A dictionary comprehension independently defines the key expression and the value expression for each iteration.

## Problem 32 — Transforming Values

### Problem

Given

```python
scores = {
    "a": 0.82,
    "b": 0.91,
    "c": 0.77,
}
```

construct a new dictionary with the same keys and percentage values:

```python
{
    "a": 82.0,
    "b": 91.0,
    "c": 77.0,
}
```

Do not modify the original.

### Solution

Iterate over the existing key-value pairs and transform only the values:

```python
percentages = {
    key: 100 * value
    for key, value in scores.items()
}
```

The original mapping remains unchanged.

#### Verification

In [32]:
scores = {
    "a": 0.82,
    "b": 0.91,
    "c": 0.77,
}

percentages = {
    key: 100 * value
    for key, value in scores.items()
}

scores, percentages

({'a': 0.82, 'b': 0.91, 'c': 0.77}, {'a': 82.0, 'b': 91.0, 'c': 77.0})

### Key Takeaway

Dictionary comprehensions can preserve keys while transforming associated values into a new mapping.

## Problem 33 — Filtering a Dictionary

### Problem

Using the scores from Problem 32, construct a new dictionary containing only scores greater than or equal to `0.80`.

Use a dictionary comprehension.

### Solution

Use a trailing filter condition:

```python
selected = {
    key: value
    for key, value in scores.items()
    if value >= 0.80
}
```

The result is

```python
{"a": 0.82, "b": 0.91}
```

#### Verification

In [33]:
scores = {
    "a": 0.82,
    "b": 0.91,
    "c": 0.77,
}

selected = {
    key: value
    for key, value in scores.items()
    if value >= 0.80
}

selected

{'a': 0.82, 'b': 0.91}

### Key Takeaway

Filtering into a new dictionary avoids structural mutation of the mapping being traversed.

## Problem 34 — Key Collision

### Problem

Predict:

```python
data = {
    "A": 1,
    "a": 2,
}

normalized = {
    key.lower(): value
    for key, value in data.items()
}
```

What is `normalized`?

Explain why one association disappears.

### Solution

Both source keys become the same key `"a"` after `.lower()`.

The first iteration produces `"a": 1`; the second produces `"a": 2`, which replaces it.

Thus:

```python
normalized == {"a": 2}
```

The collision follows the dictionary requirement that keys are unique.

#### Verification

In [34]:
data = {
    "A": 1,
    "a": 2,
}

normalized = {
    key.lower(): value
    for key, value in data.items()
}

normalized

{'a': 2}

### Key Takeaway

Transforming dictionary keys can create collisions; later associations overwrite earlier ones for the same resulting key.

## Problem 35 — Frequency Counting

### Problem

Given

```python
labels = [
    "cat",
    "dog",
    "cat",
    "bird",
    "dog",
    "cat",
]
```

build a dictionary containing the frequency of each label using

```python
counts.get(label, 0)
```

The expected result is

```python
{
    "cat": 3,
    "dog": 2,
    "bird": 1,
}
```

### Solution

Initialise an empty dictionary and update the current count:

```python
counts = {}

for label in labels:
    counts[label] = counts.get(label, 0) + 1
```

For a new label, `get(label, 0)` returns `0`. For an existing label, it returns the current count.

#### Verification

In [35]:
labels = [
    "cat",
    "dog",
    "cat",
    "bird",
    "dog",
    "cat",
]

counts = {}

for label in labels:
    counts[label] = counts.get(label, 0) + 1

counts

{'cat': 3, 'dog': 2, 'bird': 1}

### Key Takeaway

Frequency counting uses keys as categories and values as mutable running state.

## Problem 36 — Grouping

### Problem

Given

```python
records = [
    ("A", 10),
    ("B", 20),
    ("A", 30),
    ("C", 40),
    ("B", 50),
]
```

construct

```python
{
    "A": [10, 30],
    "B": [20, 50],
    "C": [40],
}
```

First solve the problem explicitly using membership tests.

Then rewrite it using `setdefault()`.

### Solution

Explicit form:

```python
groups = {}

for key, value in records:
    if key not in groups:
        groups[key] = []

    groups[key].append(value)
```

Using `setdefault()`:

```python
groups = {}

for key, value in records:
    groups.setdefault(key, []).append(value)
```

In both cases, each key identifies a list containing all associated observations.

#### Verification

In [36]:
records = [
    ("A", 10),
    ("B", 20),
    ("A", 30),
    ("C", 40),
    ("B", 50),
]

groups_1 = {}

for key, value in records:
    if key not in groups_1:
        groups_1[key] = []
    groups_1[key].append(value)

groups_2 = {}

for key, value in records:
    groups_2.setdefault(key, []).append(value)

groups_1, groups_2

({'A': [10, 30], 'B': [20, 50], 'C': [40]},
 {'A': [10, 30], 'B': [20, 50], 'C': [40]})

### Key Takeaway

Grouping requires accumulating multiple values under each key rather than allowing later values to overwrite earlier ones.

## Problem 37 — Lookup Table

### Problem

Given

```python
class_names = {
    0: "negative",
    1: "positive",
}
```

and

```python
predictions = [1, 0, 1, 1, 0]
```

construct

```python
[
    "positive",
    "negative",
    "positive",
    "positive",
    "negative",
]
```

using the dictionary as a lookup table.

### Solution

Use each prediction as a dictionary key:

```python
labels = [
    class_names[prediction]
    for prediction in predictions
]
```

The dictionary converts each numeric identifier into its corresponding label.

#### Verification

In [37]:
class_names = {
    0: "negative",
    1: "positive",
}

predictions = [1, 0, 1, 1, 0]

labels = [
    class_names[prediction]
    for prediction in predictions
]

labels

['positive', 'negative', 'positive', 'positive', 'negative']

### Key Takeaway

A dictionary is a natural lookup table when each input identifier determines one associated output value.

## Problem 38 — Selecting Model Names

### Problem

Given

```python
results = {
    "linear": 0.82,
    "forest": 0.91,
    "svm": 0.88,
    "tree": 0.74,
}
```

construct a list containing the names of models with scores at least `0.85`.

The expected result is

```python
["forest", "svm"]
```

### Solution

Iterate over key-value pairs, filter by score, and store only the keys:

```python
selected = [
    model
    for model, score in results.items()
    if score >= 0.85
]
```

#### Verification

In [38]:
results = {
    "linear": 0.82,
    "forest": 0.91,
    "svm": 0.88,
    "tree": 0.74,
}

selected = [
    model
    for model, score in results.items()
    if score >= 0.85
]

selected

['forest', 'svm']

### Key Takeaway

The output structure need not be a dictionary; mappings can be traversed to construct lists, sets, or other objects as required.

## Problem 39 — Structural Mutation during Iteration

### Problem

Explain why the following code is problematic:

```python
scores = {
    "a": 0.91,
    "b": 0.42,
    "c": 0.87,
}

for key in scores:
    if scores[key] < 0.5:
        del scores[key]
```

Rewrite the operation using a dictionary comprehension.

### Solution

The loop changes the dictionary's size while the dictionary iterator is active. Python detects this structural mutation and raises

```text
RuntimeError: dictionary changed size during iteration
```

A filtering comprehension is safer:

```python
scores = {
    key: value
    for key, value in scores.items()
    if value >= 0.5
}
```

This constructs a new dictionary instead of structurally modifying the mapping being traversed.

#### Verification

In [39]:
scores = {
    "a": 0.91,
    "b": 0.42,
    "c": 0.87,
}

try:
    for key in scores:
        if scores[key] < 0.5:
            del scores[key]
except RuntimeError as exc:
    print(type(exc).__name__, exc)

scores = {
    "a": 0.91,
    "b": 0.42,
    "c": 0.87,
}

scores = {
    key: value
    for key, value in scores.items()
    if value >= 0.5
}

scores

RuntimeError dictionary changed size during iteration


{'a': 0.91, 'c': 0.87}

### Key Takeaway

Avoid changing the size of a dictionary while iterating over that same dictionary; construct a filtered mapping or iterate over a detached snapshot.

## Problem 40 — Reasoning about References

### Problem

Without executing the code, determine the exact output:

```python
a = {
    "features": ["age", "income"],
    "metrics": {
        "accuracy": 0.91,
    },
}

b = a
c = a.copy()

b["features"].append("balance")
c["metrics"]["accuracy"] = 0.95
c["name"] = "model_a"

print(a)
print(b)
print(c)

print(a is b)
print(a is c)
print(a["features"] is c["features"])
print(a["metrics"] is c["metrics"])
```

Trace the identity relationships at both the outer and nested levels.

### Solution

Initially:

- `b` aliases `a`;
- `c` is a shallow copy of the outer dictionary;
- the nested list and nested dictionary are shared by `a`, `b`, and `c`.

Then:

1. `b["features"].append("balance")` mutates the shared list.
2. `c["metrics"]["accuracy"] = 0.95` mutates the shared nested metrics dictionary.
3. `c["name"] = "model_a"` modifies only the outer dictionary `c`.

The printed mappings are therefore:

```text
{'features': ['age', 'income', 'balance'], 'metrics': {'accuracy': 0.95}}
{'features': ['age', 'income', 'balance'], 'metrics': {'accuracy': 0.95}}
{'features': ['age', 'income', 'balance'], 'metrics': {'accuracy': 0.95}, 'name': 'model_a'}
```

The identity tests are:

```text
True
False
True
True
```

The outer copy is independent, but its nested mutable values remain shared.

#### Verification

In [40]:
a = {
    "features": ["age", "income"],
    "metrics": {
        "accuracy": 0.91,
    },
}

b = a
c = a.copy()

b["features"].append("balance")
c["metrics"]["accuracy"] = 0.95
c["name"] = "model_a"

print(a)
print(b)
print(c)

print(a is b)
print(a is c)
print(a["features"] is c["features"])
print(a["metrics"] is c["metrics"])

{'features': ['age', 'income', 'balance'], 'metrics': {'accuracy': 0.95}}
{'features': ['age', 'income', 'balance'], 'metrics': {'accuracy': 0.95}}
{'features': ['age', 'income', 'balance'], 'metrics': {'accuracy': 0.95}, 'name': 'model_a'}
True
False
True
True


### Key Takeaway

For nested mutable mappings, identity must be tracked separately at the outer-container level and at each nested value level.